In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torchaudio
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold


os.makedirs("history", exist_ok=True)
os.makedirs("models", exist_ok=True)


def set_seed(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [2]:
class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512, f_min=20, f_max=16000, channels=1, norm="slaney", mel_scale="htk", target_size=(256, 256), top_db=80.0, delta_win=5,):
        super().__init__()
        self.channels = channels
        self.top_db = top_db

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=f_min,
            f_max=f_max,
            mel_scale=mel_scale,
            pad_mode="reflect",
            power=2.0,
            norm=norm,
            center=True,
        )

        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S)) 
        if self.top_db is not None:
            max_val = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x):
        # x: (B, T) or (T,)
        squeeze = False
        if x.dim() == 1:
            x = x.unsqueeze(0)
            squeeze = True

        mel_spec = self.mel_transform(x)          # (B, n_mels, time)
        mel_spec = self.power_to_db(mel_spec)

        mel_spec = mel_spec.unsqueeze(1).repeat(1, self.channels, 1, 1)
        mel_spec = self.resize(mel_spec)           # (B, C, H, W)

        B, C = mel_spec.shape[:2]
        flat = mel_spec.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel_spec = (mel_spec - mins) / (maxs - mins + 1e-7)

        if squeeze:
            mel_spec = mel_spec.squeeze(0)

        return mel_spec

In [3]:
class BirdDataset(Dataset):
    PATH = '/kaggle/input/competitions/birdclef-2026/'

    config = {"sr": 32000, 'seed': 2, 'train_only': False}

    def __init__(self, is_train=True, fold=0, config={}, use_soundscapes=True):
        self.config.update(config)
        self.is_train = is_train

        # ── Load taxonomy (defines all 234 classes) ──────────────────────────
        tax = pd.read_csv(self.PATH + 'taxonomy.csv')
        self.LABELS = list(np.unique(tax.primary_label.dropna()))

        # ── Load train_audio data ─────────────────────────────────────────────
        self.SND_PATH = self.PATH + 'train_audio/'
        df = pd.read_csv(self.PATH + 'train.csv')
        df.index = df.filename.values

        IDX = np.unique(df.index)
        np.random.seed(self.config['seed'])
        np.random.shuffle(IDX)

        if self.config['train_only']:
            idx = IDX
        else:
            skf = StratifiedKFold(n_splits=5)
            FOLDS = list(skf.split(IDX, df.loc[IDX].primary_label.fillna('none').values))
            train_idx = IDX[FOLDS[fold][0]].tolist()
            val_idx   = IDX[FOLDS[fold][1]].tolist()
            idx = train_idx if is_train else val_idx

        DF = df.loc[idx].copy()

        # Build paths and multi-hot labels from train_audio
        # Secondary labels from train.csv (multi-hot encoding)
        paths_audio  = list(self.SND_PATH + DF['filename'].values)
        labels_audio = [self._make_multilabel(row) for _, row in DF.iterrows()]

        # ── Load train_soundscapes data (fixes 28 zero-shot species) ─────────
        # 28 species ONLY exist in soundscapes — without this they can never be predicted
        paths_sc  = []
        labels_sc = []

        if use_soundscapes and is_train:
            sc_label_path = self.PATH + 'train_soundscapes_labels.csv'
            try:
                df_sc = pd.read_csv(sc_label_path)
                self.SC_PATH = self.PATH + 'train_soundscapes/'

                for _, row in df_sc.iterrows():
                    fname    = str(row['filename']) if 'filename' in df_sc.columns else str(row['id'])
                    sc_file  = self.SC_PATH + fname
                    if not sc_file.endswith('.ogg'):
                        sc_file += '.ogg'

                    # primary label
                    primary = str(row['primary_label']) if 'primary_label' in df_sc.columns else str(row['ebird_code'])

                    # secondary labels (may not exist)
                    secondaries = []
                    if 'secondary_labels' in df_sc.columns and pd.notna(row['secondary_labels']):
                        raw = str(row['secondary_labels']).strip("[]").replace("'", "").replace("\"", "")
                        secondaries = [s.strip() for s in raw.split(',') if s.strip()]

                    label_vec = self._make_multilabel_from_parts(primary, secondaries)
                    paths_sc.append(sc_file)
                    labels_sc.append(label_vec)

                print(f"Loaded {len(paths_sc)} soundscape segments for training")
                print(f"Zero-shot species coverage: soundscapes add species not in train_audio")
            except FileNotFoundError:
                print("train_soundscapes_labels.csv not found — training on train_audio only")

        # ── Combine both datasets ─────────────────────────────────────────────
        self.paths  = paths_audio + paths_sc
        self.labels = labels_audio + labels_sc

        label_matrix       = np.stack(self.labels)
        self.class_counts  = label_matrix.sum(axis=0)

        # Report zero-shot species coverage
        if use_soundscapes and is_train and len(paths_sc) > 0:
            audio_only_counts = np.stack(labels_audio).sum(axis=0)
            sc_only_counts    = np.stack(labels_sc).sum(axis=0)   if labels_sc else np.zeros(len(self.LABELS))
            zero_shot_fixed   = np.sum((audio_only_counts == 0) & (sc_only_counts > 0))
            print(f"Zero-shot species now covered by soundscapes: {zero_shot_fixed}")

    # ── Label helpers ─────────────────────────────────────────────────────────
    def _make_multilabel(self, row):
        """Multi-hot from train.csv row (primary + secondary labels)."""
        out = np.zeros(len(self.LABELS), dtype=float)

        primary = str(row['primary_label']) if pd.notna(row.get('primary_label')) else None
        if primary and primary in self.LABELS:
            out[self.LABELS.index(primary)] = 1.0

        # secondary_labels column (space or list separated)
        sec_col = row.get('secondary_labels', None)
        if sec_col and pd.notna(sec_col):
            raw = str(sec_col).strip("[]").replace("'", "").replace("\"", "")
            for s in raw.split():
                s = s.strip().rstrip(',')
                if s in self.LABELS:
                    out[self.LABELS.index(s)] = 1.0
        return out

    def _make_multilabel_from_parts(self, primary, secondaries):
        out = np.zeros(len(self.LABELS), dtype=float)
        if primary in self.LABELS:
            out[self.LABELS.index(primary)] = 1.0
        for s in secondaries:
            if s in self.LABELS:
                out[self.LABELS.index(s)] = 1.0
        return out

    def make_labels(self, X):
        """Legacy single-label helper (kept for compatibility)."""
        out = np.zeros(len(self.LABELS)).astype(bool)
        if X in self.LABELS:
            out[self.LABELS.index(X)] = True
        return out

    def load_sound(self, filepath, DUR=5 * 32000):
        try:
            wav, sr = torchaudio.load(filepath)
            wav = wav[0]
        except Exception:
            return torch.zeros(DUR)

        l = len(wav)
        if l < DUR:
            wav2    = torch.zeros(DUR)
            s       = np.random.randint(max(1, DUR - l))
            wav2[s:s + l] = wav
            wav = wav2
        else:
            if self.is_train:
                s   = random.randint(0, l - DUR)
                wav = wav[s:s + DUR]
            else:
                wav = wav[:DUR]
        return wav

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path  = self.paths[idx]
        DUR   = int(5 * self.config['sr'])
        audio = self.load_sound(path, DUR=DUR)
        label = self.labels[idx]
        return audio, torch.tensor(label, dtype=torch.float32)


def create_dataloaders(config={'batch_size': 32, 'num_workers': 0}, fold=0):
    train_dataset = BirdDataset(is_train=True,  config=config, fold=fold, use_soundscapes=True)
    val_dataset   = BirdDataset(is_train=False, config=config, fold=fold, use_soundscapes=False)

    torch.manual_seed(config['seed'])
    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=config['num_workers'],
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=config['num_workers'],
        pin_memory=True,
        drop_last=True,
    )
    targets = {
        "labels":       train_dataset.LABELS,
        "class_counts": train_dataset.class_counts,
    }
    return train_loader, val_loader, targets

In [4]:
ds = BirdDataset()

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Loaded 1478 soundscape segments for training
Zero-shot species coverage: soundscapes add species not in train_audio
Zero-shot species now covered by soundscapes: 2


In [5]:
from sklearn.metrics import roc_auc_score

def AUC(targets, outputs, verbose=False):
    targets = (targets>0).astype(float)
    num_classes = targets.shape[1]
    scored_classes = (np.sum(targets,axis=0) > 0)
    auc = roc_auc_score(targets[:,scored_classes], outputs[:,scored_classes], average='macro')
    return auc

In [6]:
from IPython.display import clear_output
from tqdm import tqdm
import datetime
import hashlib
import time

CFG = {
    'seed':2,
    "batch_size":32, 
    "num_workers":2,
    "train_only":False,
    "lr":5e-4,
    "loss":nn.BCEWithLogitsLoss(),
    'backbone_pooling':'avg', 
    'backbone':'tf_efficientnetv2_b0',
    'dropout':.2,
    'verbose':2,
    'mel':{'n_mels':256, 'f_min':20, 'n_fft':2048, 'target_size':(256,256), 'mel_scale':'slaney', 'norm':'slaney'},
    'metrics':[AUC],
    'scheduler':True,
    'model': None,
    'num_labels':234
}

class Trainer:
    
    def __init__(self, config={}, fold=0, epochs=16):
        self.config = CFG.copy()
        self.config.update(config)
        set_seed(self.config['seed'])

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.exp_id = hashlib.sha256(str(time.time()).encode()).hexdigest()

        cols = ['id', 'epoch', 'train_loss', 'val_loss', 'lr', 'timestamp', 'fold'] + \
               ['val_'+m.__name__ for m in self.config['metrics']] + \
               list(self.config.keys())
        self.history = pd.DataFrame([], columns=cols)

        self.batch_size = self.config['batch_size']
        self.fold = fold
        
        self.mel = Spectrogram(**config['mel']).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

    def train_one_epoch(self, epoch=0):
        self.model.train()        
        Loss = 0
        n_steps = len(self.train_loader)
        
        if self.config['verbose']==2:
            pbar = tqdm(enumerate(self.train_loader), total=n_steps, desc="Training")
        else:
            pbar = enumerate(self.train_loader)

        mix = Mixup(**self.config['mix'])
        
        for batch_idx, (x, y) in pbar:
            set_seed(self.config['seed']+2048*epoch+batch_idx)
            self.optimizer.zero_grad()
            
            x = x.to(self.device)
            y = y.to(self.device)
            x = self.mel(x)
            x, y = mix(x, y)
            logits = self.model(x)
            L = self.loss_fn(logits, y)
            L.backward()
            self.optimizer.step()
            Loss += L.detach().item()        
            
        return Loss / n_steps

    def validate(self):
        self.model.eval()
        Loss = 0

        n_steps = len(self.val_loader)
        out_shape = (n_steps * self.batch_size, self.config['num_labels'])
        
        Outputs = np.zeros(out_shape)
        Targets = np.zeros(out_shape)

        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(self.val_loader):
                x = x.to(self.device)
                y = y.to(self.device)
                x = self.mel(x)
                logits = self.model(x)
                
                L = self.loss_fn(logits, y)
                Loss += L.detach().item()

                start = batch_idx * self.batch_size
                end   = start + self.batch_size
                Outputs[start:end] = logits.sigmoid().cpu().numpy()
                Targets[start:end] = y.cpu().numpy()

        scores = []
        for metric in self.config['metrics']:
            try:
                score = metric(Targets, Outputs)
            except Exception:
                score = 0.0
            scores.append(score)

        return scores, Loss / n_steps

    def train(self, epochs=16, checkpoint_freq='end'):
        self.train_loader, self.val_loader, self.targets = create_dataloaders(
            self.config, fold=self.fold)

        self.model     = self.config['model'](config=self.config)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.config['lr'])
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, epochs, eta_min=1e-8)
        self.loss_fn   = self.config['loss']
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model  = self.model.to(self.device)
        
        best = (np.inf, 0, 0)
            
        for epoch in range(epochs):
            torch.manual_seed(self.config['seed'] + epoch)
            
            train_loss = self.train_one_epoch(epoch=epoch)
            
            if not self.config['train_only']:
                val_scores, val_loss = self.validate()
                if val_loss < best[0]:
                    best = (val_loss, val_scores, epoch)
            else:
                val_scores = [None] * len(self.config['metrics'])
                val_loss   = None

            self.history.loc[len(self.history)] = [
                self.exp_id, epoch, train_loss, val_loss,
                self.optimizer.param_groups[0]['lr'],
                str(datetime.datetime.now()), self.fold,
                *val_scores
            ] + list(self.config.values())

            self.history.to_csv(f'history/{self.exp_id}.csv', index=False)

            if checkpoint_freq == 'epoch':
                torch.save(self.model.state_dict(), f"models/{self.exp_id}_{epoch}.pth")

            if self.config['scheduler']:
                self.scheduler.step()

            clear_output(wait=False)
            print(self.exp_id, '\n')
            print(f"\033[1m Epoch {epoch+1}/{epochs}")
            print(f'\033[1m Training \t|\t loss={np.round(train_loss, 3)}\033[0m')
            if not self.config['train_only']:
                score_str = '  -  '.join(
                    [f"{m.__name__}={np.round(s, 3)}" 
                     for m, s in zip(self.config['metrics'], val_scores)])
                print(f'\033[1m Validation \t|\t loss={np.round(val_loss, 3)}  -  {score_str}\033[0m')
                print(f'\033[1m Best : {score_str} at epoch {best[2]}\033[0m')

        torch.save(self.model.state_dict(), f"models/{self.exp_id}_best.pth")
        return self.model

In [7]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            inputs,
            targets,
            reduction='none'
        )

        pt = torch.exp(-bce)

        focal_loss = (
            self.alpha *
            (1 - pt) ** self.gamma *
            bce
        )

        return focal_loss.mean()


def MacroF1(targets, outputs, threshold=0.5):
    from sklearn.metrics import f1_score

    targets_bin = (targets > 0).astype(int)
    outputs_bin = (outputs > threshold).astype(int)

    scored = targets_bin.sum(axis=0) > 0

    return f1_score(
        targets_bin[:, scored],
        outputs_bin[:, scored],
        average='macro',
        zero_division=0
    )


def RareSpeciesF1(
        targets,
        outputs,
        class_counts,
        rare_threshold=20,
        threshold=0.5):

    from sklearn.metrics import f1_score

    rare_classes = np.where(
        class_counts < rare_threshold
    )[0]

    if len(rare_classes) == 0:
        return 0

    targets_bin = (targets > 0).astype(int)
    outputs_bin = (outputs > threshold).astype(int)

    return f1_score(
        targets_bin[:, rare_classes],
        outputs_bin[:, rare_classes],
        average='macro',
        zero_division=0
    )

In [8]:
import timm

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = {
            'scale':1,
            'backbone_pooling':'avg',
            'backbone':'tf_efficientnetv2_b0',
            'dropout':0.1,
            'pretrained':True,
            'channels':1,
            'num_labels':234,
        }
        if config: self.config.update(config)

        self.training = True

        self.backbone = timm.create_model(
            self.config['backbone'], 
            pretrained=self.config['pretrained'],  
            num_classes=self.config['num_labels'],  
            global_pool=self.config['backbone_pooling'],
            in_chans=self.config['channels'],
            drop_rate=self.config['dropout'],
        )
        feature_dim = self.backbone.num_features
        print(self.config['num_labels'])
        
    def forward(self, x):
        labels = self.backbone(x)
        return labels

In [9]:
class Mixup(nn.Module):
    def __init__(self, alpha=0.5, theta=1):
        super().__init__()
        self.alpha = alpha
        self.theta = theta

    def forward(self, x, y):
        batch_size = x.size(0)
        
        lam = torch.tensor(np.random.beta(self.alpha,self.alpha, batch_size)).to(x.device)
        lam = torch.maximum(lam, 1-lam).float()
        
        idx = torch.randperm(batch_size).to(x.device)

        x = lam[...,None, None, None]* x + (1 - lam)[...,None, None, None] * x[idx]
        if isinstance(y, list): 
            for i in range(len(y)):
                y[i] = lam[...,None] * y[i] + (1 - lam)[...,None] * y[i][idx]
                y[i][y[i]>=self.theta] = 1
        else: 
            y = lam[...,None] * y + (1 - lam)[...,None] * y[idx]
            y[y>=self.theta] = 1
        
        return x, y

In [10]:
from sklearn.metrics import f1_score

# ── Zero-shot species analysis ────────────────────────────────────────────────
# 28 species ONLY appear in soundscapes — impossible to predict without them
print("Checking zero-shot species coverage...")
train_df   = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train.csv')
tax_df     = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
ALL_LABELS = list(np.unique(tax_df.primary_label.dropna()))

audio_species = set(train_df['primary_label'].dropna().unique())
try:
    sc_df = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
    sc_col = 'primary_label' if 'primary_label' in sc_df.columns else 'ebird_code'
    sc_species = set(sc_df[sc_col].dropna().unique())
    zero_shot  = sc_species - audio_species
    print(f"train_audio only:   {len(audio_species)} species")
    print(f"Both sets:          {len(audio_species & sc_species)} species")
    print(f"Soundscapes only:   {len(zero_shot)} species  <-- ZERO-SHOT")
    print(f"Zero-shot species: {zero_shot}")
except FileNotFoundError:
    print("Soundscape labels not found")

# ── Metrics ───────────────────────────────────────────────────────────────────
def MacroF1(targets, outputs, threshold=0.5):
    targets_bin = (targets > 0).astype(int)
    outputs_bin = (outputs > threshold).astype(int)
    scored = targets_bin.sum(axis=0) > 0
    return f1_score(targets_bin[:, scored], outputs_bin[:, scored],
                    average='macro', zero_division=0)

def RareSpeciesF1(targets, outputs, class_counts=None, rare_threshold=20, threshold=0.5):
    if class_counts is None:
        return 0.0
    rare_classes = np.where(class_counts < rare_threshold)[0]
    if len(rare_classes) == 0:
        return 0.0
    targets_bin = (targets > 0).astype(int)
    outputs_bin = (outputs > threshold).astype(int)
    return f1_score(targets_bin[:, rare_classes], outputs_bin[:, rare_classes],
                    average='macro', zero_division=0)

# ── Config ────────────────────────────────────────────────────────────────────
# KEY CHANGES vs baseline:
# 1. FocalLoss instead of BCEWithLogitsLoss  -> focuses on hard/rare classes
# 2. use_soundscapes=True in BirdDataset    -> fixes 28 zero-shot species
# 3. Multi-hot labels                        -> secondary species also supervised
# 4. MacroF1 + RareSpeciesF1 metrics        -> actually measure rare class performance

config = {
    'seed':           2,
    'batch_size':     32,
    "num_workers":    2,
    "backbone":       "tf_efficientnetv2_b0",
    "loss":           FocalLoss(alpha=1, gamma=2),   # Focal Loss for class imbalance
    'mel':            {'n_mels': 256, 'f_min': 20, 'n_fft': 2048, 'target_size': (256, 256)},
    "mix":            {"alpha": 0.5, "theta": 0.8},
    "pretrained":     True,
    "model":          BirdModel,
    'train_only':     False,
    'metrics':        [AUC, MacroF1],               # AUC + MacroF1 both tracked
    'use_soundscapes': True,                         # Include soundscape data
}

trainer = Trainer(config=config)
model   = trainer.train(epochs=16)

# Save model
torch.save(model.state_dict(), '/kaggle/working/focal_soundscape_model.pth')
print("Model saved to /kaggle/working/focal_soundscape_model.pth")

6c3a46578cc7588f93c3fb6d1ad61210480862a3cffd6006fbfc41831d12382b 

 Epoch 16/16
 Training 	|	 loss=0.003
 Validation 	|	 loss=0.004  -  AUC=0.965  -  MacroF1=0.664
 Best : AUC=0.965  -  MacroF1=0.664 at epoch 13
Model saved to /kaggle/working/focal_soundscape_model.pth


In [11]:
import pandas as pd
import glob

# Find the history file
files = glob.glob('history/*.csv')
print(files)

# Read and display it
df = pd.read_csv(files[0])
print(df[['epoch', 'train_loss', 'val_loss', 'val_AUC']].to_string())

['history/6c3a46578cc7588f93c3fb6d1ad61210480862a3cffd6006fbfc41831d12382b.csv']
    epoch  train_loss  val_loss   val_AUC
0       0    0.008545  0.006215  0.884545
1       1    0.005549  0.004945  0.933711
2       2    0.004841  0.004277  0.949018
3       3    0.004447  0.004046  0.954762
4       4    0.004166  0.003828  0.959210
5       5    0.003923  0.003757  0.959726
6       6    0.003735  0.003648  0.963089
7       7    0.003504  0.003553  0.962489
8       8    0.003355  0.003556  0.963283
9       9    0.003209  0.003526  0.964538
10     10    0.003027  0.003575  0.965449
11     11    0.002901  0.003538  0.966038
12     12    0.002802  0.003532  0.964940
13     13    0.002719  0.003486  0.965798
14     14    0.002686  0.003528  0.965346
15     15    0.002656  0.003537  0.965101
